In [2]:

import os
# os.environ["HF_HUB_OFFLINE"] = "1"

In [1]:
from huggingface_hub import snapshot_download
from datasets import load_dataset
from gptqmodel import GPTQModel, QuantizeConfig

WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.1.0+dd6b020
Transformers : 5.12.1
Torch        : 2.11.0+cu130
Triton       : 3.6.0


In [10]:
# 1. Define paths
model_id = "meta-llama/Llama-3.2-3B-Instruct"
quant_path = "./models/llama-3.2-3b-instruct-gptq-custom"

In [4]:
print("Locating cached model and skipping unnecessary files...")
# 2. Safely resolve the local cache path, explicitly ignoring the 6.4GB raw Meta weights
local_model_path = snapshot_download(
    repo_id=model_id,
    ignore_patterns=["original/*", "*.pth"] # <--- THIS IS THE MAGIC FIX
)

Locating cached model and skipping unnecessary files...


INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct/revision/main "HTTP/1.1 200 OK"


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

In [5]:
# 2. Define the GPTQ configuration
# group_size=128 and bits=4 are the standard for 4-bit quantization
quant_config = QuantizeConfig(bits=4, group_size=128)

INFO  QuantizeConfig: offload_to_disk_path auto set to temporary dir `/tmp/gptqmodel_3oacz_f9`


In [6]:
print("Downloading calibration dataset...")
# 3. Load a calibration dataset (Required for GPTQ)
# Here 1024 samples to give the algorithm enough data to observe activation patterns
calibration_dataset = load_dataset(
    "allenai/c4", 
    data_files="en/c4-train.00001-of-01024.json.gz", 
    split="train",
).select(range(1024))["text"]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/allenai/c4/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/allenai/c4/1588ec454efa1a09f29cd18ddd04fe05fc8653a2/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/allenai/c4/resolve/1588ec454efa1a09f29cd18ddd04fe05fc8653a2/c4.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/allenai/c4/allenai/c4.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/allenai/c4/revision/1588ec454efa1a09f29cd18ddd04fe05fc8653a2 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/allenai/c4/resolve/1588ec454efa1a09f29cd18ddd04fe05fc8653a2/.huggingface.yaml "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/allenai/c4/tree/1588ec454efa1a09f29c

In [7]:
print(f"Loading base model from: {local_model_path}")
# 4. Load the model for quantization using GPTQModel
model = GPTQModel.load(local_model_path, quant_config)

Loading base model from: /media/yedhu/HDDStorage/Workspace/ai-ml/models/huggingface/hub/models--meta-llama--Llama-3.2-3B-Instruct/snapshots/0cb88a4f764b7a12671c53f0838cd831a0843b95


INFO  Loader: Auto dtype (native bfloat16): `torch.bfloat16`                   


INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]


INFO  Loader: using checkpoint-backed lazy turtle source for `/media/yedhu/HDDStorage/Workspace/ai-ml/models/huggingface/hub/models--meta-llama--Llama-3.2-3B-Instruct/snapshots/0cb88a4f764b7a12671c53f0838cd831a0843b95`


INFO:tokenicer.tokenicer:Tokenicer: Auto fixed pad_token_id=128004 (token='<|finetune_right_pad_id|>').


INFO  Model: Loaded `generation_config`: GenerationConfig {
  "bos_token_id": 128000,
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "output_attentions": false,
  "output_hidden_states": false,
  "use_cache": true
}



INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.


INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 128000,
  "do_sample": true,
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "temperature": 0.6,
  "top_p": 0.9
}



INFO  Kernel: loaded -> `[]`                                                   


In [8]:
print("Starting the GPTQ quantization process (this will take time)...")
# 5. Quantize using the dataset
# You can increase batch_size if your GPU has plenty of VRAM to speed this up
model.quantize(calibration_dataset, batch_size=2)

Starting the GPTQ quantization process (this will take time)...


INFO  Packing Kernel: selected: `TorchLinear`                                  


INFO  Packing Kernel: selected: `TorchLinear`                                  


INFO  Calibration: Sort in descending order by length                          


INFO  Calibration: Total padded tokens: 4466                                   


...

INFO  | Finalize offload   | 196    | 0.007  | 0.033  | 6.431   | 0.8%   | model.layers.27.mlp.gate_proj                     |


INFO  +--------------------+--------+--------+--------+---------+--------+---------------------------------------------------+


INFO  | Process finalize   | 1      | 0.012  | 0.012  | 0.012   | 0.0%   | gptq                                              |


INFO  +--------------------+--------+--------+--------+---------+--------+---------------------------------------------------+


{'gptq': [{'process': 'gptq',
   'layer': 0,
   'module': 'self_attn.q_proj',
   'feat: in, out': '3072, 3072',
   'dtype: size': 'bf16: 18.6MB',
   'loss': '0.0000030229',
   'samples': '454766',
   'damp': '0.05000',
   'time': '0.843',
   'fwd_time': '2.018',
   '(v)ram': 'cuda 2.77G'},
  {'process': 'gptq',
   'layer': 0,
   'module': 'self_attn.k_proj',
   'feat: in, out': '3072, 1024',
   'dtype: size': 'bf16: 6.2MB',
   'loss': '0.0000015400',
   'samples': '454766',
   'damp': '0.05000',
   'time': '0.880',
   'fwd_time': '2.018',
   '(v)ram': 'cuda 2.77G'},
  {'process': 'gptq',
   'layer': 0,
   'module': 'self_attn.v_proj',
   'feat: in, out': '3072, 1024',
   'dtype: size': 'bf16: 6.2MB',
   'loss': '0.0000000859',
   'samples': '454766',
   'damp': '0.05000',
   'time': '0.892',
   'fwd_time': '2.018',
   '(v)ram': 'cuda 2.77G'},
  {'process': 'gptq',
   'layer': 0,
   'module': 'self_attn.o_proj',
   'feat: in, out': '3072, 3072',
   'dtype: size': 'bf16: 18.6MB',
   'los

In [11]:
print(f"Saving quantized weights and tokenizer to local folder: {quant_path}")
# 6. Save the fully quantized model
model.save(quant_path)

Saving quantized weights and tokenizer to local folder: ./models/llama-3.2-3b-instruct-gptq-custom


Writing model shards: 0it [00:00, ?it/s]

INFO  Saved Quantize Config: 
{
  "bits": 4,
  "group_size": 128,
  "desc_act": false,
  "lm_head": false,
  "method": "gptq",
  "quant_method": "gptq",
  "format": "gptq",
  "checkpoint_format": "gptq",
  "pack_dtype": "int32",
  "meta": {
    "quantizer": [
      "gptqmodel:7.1.0"
    ],
    "uri": "https://github.com/modelcloud/gptqmodel",
    "damp_percent": 0.05,
    "damp_auto_increment": 0.01,
    "static_groups": false,
    "true_sequential": true,
    "mse": 0.0,
    "gptaq": null,
    "foem": null,
    "act_group_aware": true,
    "fallback": {
      "strategy": "rtn",
      "threshold": "0.5%",
      "smooth": null
    },
    "offload_to_disk": true,
    "offload_to_disk_path": "/tmp/gptqmodel_3oacz_f9",
    "pack_impl": "cpu",
    "gc_mode": "interval",
    "wait_for_submodule_finalizers": false,
    "auto_forward_data_parallel": true,
    "dense_vram_strategy": "exclusive",
    "dense_vram_strategy_devices": null,
    "moe_vram_strategy": "exclusive",
    "moe_vram_strateg

Files in directory:
config.json
quant_log.csv
quantize_config.json
generation_config.json
Content of saved `generation_config.json`:
{
    "bos_token_id": 128000,
    "do_sample": true,
    "eos_token_id": [
        128001,
        128008,
        128009
    ],
    "temperature": 0.6,
    "top_p": 0.9,
    "transformers_version": "5.12.1"
}
Content of saved `config.json`:
{
    "architectures": [
        "LlamaForCausalLM"
    ],
    "attention_bias": false,
    "attention_dropout": 0.0,
    "bos_token_id": 128000,
    "dtype": "bfloat16",
    "eos_token_id": [
        128001,
        128008,
        128009
    ],
    "head_dim": 128,
    "hidden_act": "silu",
    "hidden_size": 3072,
    "initializer_range": 0.02,
    "intermediate_size": 8192,
    "max_position_embeddings": 131072,
    "mlp_bias": false,
    "model_type": "llama",
    "num_attention_heads": 24,
    "num_hidden_layers": 28,
    "num_key_value_heads": 8,
    "pad_token_id": 128004,
    "pretraining_tp": 1,
    "quantiz

INFO  Module: Re-tied embedding weights on shell model after lazy sync         


...

INFO  +--------------------+--------+--------+--------+---------+--------+-------------------------------------------------------------------------------------------------------------+


INFO  | Model save         | 2      | 1.155  | 1.183  | 2.365   | 0.3%   | /home/yedhu/workspace/ai-ml/projects/awq-gptq-pipelines/notebooks/models/llama-3.2-3b-instruct-gptq-custom  |


INFO  +--------------------+--------+--------+--------+---------+--------+-------------------------------------------------------------------------------------------------------------+


INFO  | Process finalize   | 1      | 0.012  | 0.012  | 0.012   | 0.0%   | gptq                                                                                                        |


INFO  +--------------------+--------+--------+--------+---------+--------+-------------------------------------------------------------------------------------------------------------+
